![Logo SENATI](logo_senati.png)

**SENATI | Escuela de Tecnologías de la Información**  
**Taller: Machine Learning y Deep Learning**  
*Instructor: Ing. Gunar Castro Iñigo*

---

# Sistema de estimación de costo de transporte agrícola — caso práctico intermedio

**Contexto:** Un transportista quiere estimar el costo de un viaje a partir de distancia, peso y tipo de vía. Datos simulados inspirados en el transporte agrícola regional.

**Objetivo:** aplicar regresión múltiple desde lectura de datos hasta una web pequeña. Aquí no se entrega el resultado completo: debes demostrar lo que aprendiste durante los cuatro proyectos.

## Paso 1. Explorar el CSV
Busca las columnas de entrada `X`: distancia_km, carga_kg, via_no_asfaltada, y la respuesta `y`: `costo_transporte_soles`.

**¿Qué podemos preguntar al grupo?** ¿Qué significa cada variable en la vida real? ¿Cuál sería el peligro de usar esta predicción sin verificarla?

**Variables:** `distancia_km` (km recorridos), `carga_kg` (peso transportado), `via_no_asfaltada` (0 = asfaltada, 1 = no asfaltada) → `costo_transporte_soles` (costo del viaje en S/). El peligro de usar la predicción sin verificarla: los datos son simulados y no incluyen combustible, peajes ni clima; podría cotizarse por debajo del costo real y perder dinero.

In [1]:
import pandas as pd

datos = pd.read_csv('datos.csv', encoding='utf-8-sig')
print(datos.head())
print(datos.shape)
print('\nAusentes por columna:')
print(datos.isna().sum())
print('\nDuplicados:', datos.duplicated().sum())
print(datos.describe().round(2))

   viaje  distancia_km  carga_kg  via_no_asfaltada  costo_transporte_soles
0  R-001         140.1      1775                 0                  592.32
1  R-002         131.9       425                 1                  442.18
2  R-003         102.8      1525                 0                  478.20
3  R-004          50.5       325                 1                  221.73
4  R-005          67.1       975                 0                  314.59
(130, 5)

Ausentes por columna:
viaje                     0
distancia_km              0
carga_kg                  0
via_no_asfaltada          0
costo_transporte_soles    0
dtype: int64

Duplicados: 0
       distancia_km  carga_kg  via_no_asfaltada  costo_transporte_soles
count        130.00    130.00            130.00                  130.00
mean          84.52    999.04              0.52                  377.97
std           46.70    569.06              0.50                  142.02
min            3.00     50.00              0.00               

## Paso 2. Entrenar (completar)
Adapta el procedimiento de las prácticas 2 y 4. No copies resultados de otros compañeros: explica qué variables usaste, cuántas filas reservaste y cómo interpretas el MAE.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

columnas = ['distancia_km', 'carga_kg', 'via_no_asfaltada']
objetivo = 'costo_transporte_soles'
X = datos[columnas]
y = datos[objetivo]

# 80 % entrenamiento / 20 % prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Entrenamiento:', len(X_train), 'filas | Prueba:', len(X_test), 'filas')

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print('\nIntercepto b0 =', round(modelo.intercept_, 4))
for n, k in zip(columnas, modelo.coef_):
    print(f'{n:<28} = {k:.4f}')

pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print(f'\nMAE (prueba) = {mae:.2f} soles')
print(f'R²  (prueba) = {r2:.3f}')

Entrenamiento: 104 filas | Prueba: 26 filas

Intercepto b0 = 21.5196
distancia_km                 = 2.7742
carga_kg                     = 0.1073
via_no_asfaltada             = 26.4419

MAE (prueba) = 17.79 soles
R²  (prueba) = 0.977


### Interpretación de coeficientes
- **Intercepto (≈ 21.5):** costo fijo base teórico (S/) con distancia y carga cero.
- **distancia_km (≈ 2.77):** cada km adicional suma ~S/ 2.77 al costo.
- **carga_kg (≈ 0.107):** cada 100 kg extra suman ~S/ 10.7.
- **via_no_asfaltada (≈ 26.4):** una ruta no asfaltada añade ~S/ 26 frente a una asfaltada, con igual distancia y carga.

### Interpretación de métricas (sobre las 26 filas de prueba)
- **MAE ≈ 17.8:** el modelo se equivoca en promedio por unos **S/ 18** por viaje (costo medio ~S/ 380, ≈ 5 %).
- **R² ≈ 0.98:** explica cerca del **98 %** de la variación del costo; el ajuste es muy bueno.

In [3]:
import json
json.dump({'intercepto': float(modelo.intercept_),
           'coeficientes': {n: float(k) for n, k in zip(columnas, modelo.coef_)},
           'metricas': {'mae': float(mae), 'r2': float(r2)}},
          open('modelo_notebook.json', 'w'), indent=4)
print('Guardado modelo_notebook.json')

Guardado modelo_notebook.json


## Paso 3. Aplicación web
Tu página necesita título realista, 3 campos con unidades, botón “Calcular”, resultado y una advertencia de estimación. Usa Flask, HTML y CSS (sin JavaScript). Separa los archivos en las carpetas habituales.

## Paso 4. Pruebas obligatorias
- Viaje 25 km, 400 kg, ruta asfaltada
- Viaje 100 km, 1200 kg, ruta no asfaltada

¿Qué valores devolvió tu sistema? ¿Tienen sentido? ¿En qué circunstancias la predicción podría fallar?

In [4]:
casos = [('Viaje 25 km, 400 kg, asfaltada', 25, 400, 0),
         ('Viaje 100 km, 1200 kg, no asfaltada', 100, 1200, 1)]
for nombre, a, b, k in casos:
    p = modelo.predict(pd.DataFrame([[a, b, k]], columns=columnas))[0]
    print(f'{nombre}: S/ {p:.2f}')

Viaje 25 km, 400 kg, asfaltada: S/ 133.79
Viaje 100 km, 1200 kg, no asfaltada: S/ 454.13


**Resultados:** Viaje 1 ≈ **S/ 133.79** y viaje 2 ≈ **S/ 454.13**. Tienen sentido: el viaje largo, pesado y por trocha cuesta mucho más.

**¿Cuándo puede fallar?** Fuera de 3–165 km o 50–2050 kg, con alzas de combustible, peajes, clima o carreteras bloqueadas, carga perecible que exija refrigeración, o viajes de retorno vacío que el modelo no distingue.

## Evidencias y evaluación
Entregar notebook ejecutado, CSV, script de entrenamiento, `modelo.json`, aplicación Flask, dos capturas de prueba y conclusiones propias. Criterios: datos 3 pts; modelo 5 pts; métricas 4 pts; aplicación 5 pts; documentación 3 pts.

## Conclusiones
1. El modelo lineal ajusta muy bien (R² ≈ 0.98) con un error medio de ~S/ 18 por viaje.
2. La distancia es el principal motor del costo; la vía no asfaltada añade ~S/ 26 fijos.
3. Con 130 viajes simulados y una sola partición 80/20, las métricas son orientativas; con datos reales habría que revalidar.
4. No considera combustible variable, peajes, tiempos de espera ni interacciones (p. ej. trocha × peso).
5. La app restringe los rangos para no extrapolar fuera de lo entrenado.